<a href="https://colab.research.google.com/github/MohitSawarkar/GenAi-/blob/main/GenAiPractical4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense

# 1. Define hyperparameters
num_samples = 10000  # Number of training samples
seq_length = 6       # Length of input sequence
vocab_size = 10      # Numbers 0-9
latent_dim = 128     # Number of latent dimensions for the LSTM
START_TOKEN = 10     # Indicates the start of the decoding process
END_TOKEN = 11       # Indicates the end of the decoding process
num_tokens = 12      # Total vocabulary size (0-9 + START + END)

print("Generating synthetic data...")
# Input data: Random sequences of numbers 0-9
X = np.random.randint(0, vocab_size, size=(num_samples, seq_length))
# Target data: Reversed sequences
Y = np.flip(X, axis=1)

# Initialize arrays for one-hot encoding
encoder_input_data = np.zeros((num_samples, seq_length, num_tokens), dtype='float32')
decoder_input_data = np.zeros((num_samples, seq_length + 1, num_tokens), dtype='float32')
decoder_target_data = np.zeros((num_samples, seq_length + 1, num_tokens), dtype='float32')

# Populate one-hot encoded arrays
for i, (input_seq, target_seq) in enumerate(zip(X, Y)):
    for t, num in enumerate(input_seq):
        encoder_input_data[i, t, num] = 1.0

    decoder_input_data[i, 0, START_TOKEN] = 1.0 # Decoder input starts with START_TOKEN

    for t, num in enumerate(target_seq):
        decoder_input_data[i, t + 1, num] = 1.0
        decoder_target_data[i, t, num] = 1.0    # Target is ahead by one timestep

    decoder_target_data[i, len(target_seq), END_TOKEN] = 1.0 # Target ends with END_TOKEN

print("Data generation complete. Shapes:")
print(f"Encoder Input: {encoder_input_data.shape}")
print(f"Decoder Input: {decoder_input_data.shape}")
print(f"Decoder Target: {decoder_target_data.shape}")

Generating synthetic data...
Data generation complete. Shapes:
Encoder Input: (10000, 6, 12)
Decoder Input: (10000, 7, 12)
Decoder Target: (10000, 7, 12)


In [4]:
# 2. Build the Encoder
encoder_inputs = Input(shape=(None, num_tokens), name="encoder_inputs")
encoder_lstm = LSTM(latent_dim, return_state=True, name="encoder_lstm")
encoder_outputs, state_h, state_c = encoder_lstm(encoder_inputs)

# We discard encoder_outputs and only keep the states.
encoder_states = [state_h, state_c]

# 3. Build the Decoder
decoder_inputs = Input(shape=(None, num_tokens), name="decoder_inputs")
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, name="decoder_lstm")

# Set up the decoder, using encoder_states as initial state.
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = Dense(num_tokens, activation='softmax', name="decoder_dense")
decoder_outputs = decoder_dense(decoder_outputs)

# 4. Define the overall model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None, 12)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None, 12)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 128),     │     72,192 │ encoder_inputs[0… │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │     72,192 │ decoder_inputs[0… │
│                     │ 128), (None,      │            │ encoder_lstm[0][… │
│                     │ 128), (None,      │            │ encoder_lstm[0][… │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, None, 12)  │      1,548 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 145,932 (570.05 KB)

 Trainable params: 145,932 (570.05 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# 5. Train the model
epochs = 40
batch_size = 64

print("Starting training...")
history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.2
)
print("Training finished!")

Starting training...
Epoch 1/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.3722 - loss: 1.7627 - val_accuracy: 0.4780 - val_loss: 1.3189
Epoch 2/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.5810 - loss: 1.1040 - val_accuracy: 0.7177 - val_loss: 0.9169
Epoch 3/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.8374 - loss: 0.6035 - val_accuracy: 0.9296 - val_loss: 0.4039
Epoch 4/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.9612 - loss: 0.2901 - val_accuracy: 0.9779 - val_loss: 0.2110
Epoch 5/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.9872 - loss: 0.1598 - val_accuracy: 0.9929 - val_loss: 0.1247
Epoch 6/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.9949 - loss: 0.0991 - val_accuracy: 0.9929 - val_loss: 0.0880
Epoch 7/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - accuracy: 0.9976 - loss: 0.0663 - val_accuracy: 0.9979 - val_loss: 0.0562
Epoch 8/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.9989 - lo

In [6]:
# 6. Setup Inference Models (Sampling)

# Encoder inference model: Takes input sequence and outputs states
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder inference model: Takes previous states and previous token as input
decoder_state_input_h = Input(shape=(latent_dim,), name="input_h")
decoder_state_input_c = Input(shape=(latent_dim,), name="input_c")
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs
)
decoder_states_inf = [state_h_inf, state_c_inf]
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)
print("Inference models built successfully.")

Inference models built successfully.


In [10]:

# 7. Decoding function for new sequences
def decode_sequence(input_seq_array):
    # One-hot encode the input array
    input_seq = np.zeros(
        (1, len(input_seq_array), num_tokens),
        dtype='float32'
    )

    for t, num in enumerate(input_seq_array):
        input_seq[0, t, num] = 1.0

    # Encode the input to get the initial states for the decoder
    states_value = encoder_model.predict(input_seq, verbose=0)

    # Initialize target sequence with START_TOKEN
    target_seq = np.zeros((1, 1, num_tokens), dtype='float32')
    target_seq[0, 0, START_TOKEN] = 1.0

    stop_condition = False
    decoded_sequence = []

    while not stop_condition:
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value,
            verbose=0
        )

        # Sample the token with the highest probability
        sampled_token_index = int(
            np.argmax(output_tokens[0, -1, :])
        )

        # Stop if END_TOKEN is found or maximum length is reached
        if sampled_token_index == END_TOKEN:
            stop_condition = True
        elif len(decoded_sequence) >= seq_length:
            stop_condition = True
        else:
            decoded_sequence.append(sampled_token_index)

        # Update target sequence
        target_seq = np.zeros((1, 1, num_tokens), dtype='float32')
        target_seq[0, 0, sampled_token_index] = 1.0

        # Update decoder states
        states_value = [h, c]

    return decoded_sequence


# 8. Test on a few examples
test_sequences = [
    [1, 2, 3, 4, 5, 6],
    [9, 8, 7, 0, 1, 2],
    [5, 5, 2, 3, 9, 1]
]

print("--- Seq2Seq Reversal Test ---")

for seq in test_sequences:
    predicted = decode_sequence(seq)

    print(f"Input Sequence:    {seq}")
    print(f"Predicted Output:  {predicted}")
    print("-" * 30)



--- Seq2Seq Reversal Test ---
Input Sequence:    [1, 2, 3, 4, 5, 6]
Predicted Output:  [6, 5, 4, 3, 2, 1]
------------------------------
Input Sequence:    [9, 8, 7, 0, 1, 2]
Predicted Output:  [2, 1, 0, 7, 8, 9]
------------------------------
Input Sequence:    [5, 5, 2, 3, 9, 1]
Predicted Output:  [1, 9, 3, 2, 5, 5]
------------------------------
